# Sparse-structure coordinate-head comparison

For every held-out object, this notebook compares the ground-truth mesh, the Objective-1 prediction, and the decoder-aware sparse-structure coordinate-head prediction. All three meshes use the same cut plane and camera.

The notebook prefers the canonical ground-truth PLY from the processed dataset. If that dataset is not present locally, it loads the raw ShapeNet OBJ and applies the same rotation and normalization used by `render_kiui.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
import trimesh
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/ss_coordinate_head_v2/eval_test")
DATASET_DIR = Path("datasets/ShapeNetTRELLIS_full/test")
RAW_SHAPENET_DIR = Path("ShapeNet")
SEED = 42

# Remove the camera-facing portion along this axis.
# Use [0, 1, 0] or [0, 0, 1] to inspect another direction.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.5  # 0.5 removes half of the object

METHODS = [
    ("Ground truth", None),
    ("Objective 1", "objective1"),
    ("Coordinate head", "coordinate_head"),
]

mesh_dirs = {
    label: RESULTS_DIR / "predictions" / method / f"seed_{SEED}" / "mesh"
    for label, method in METHODS if method is not None
}
sample_ids = [
    line.strip()
    for line in (RESULTS_DIR / "selected_ids.txt").read_text().splitlines()
    if line.strip()
]

for sample_id in sample_ids:
    for label, mesh_dir in mesh_dirs.items():
        path = mesh_dir / f"{sample_id}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing {label} mesh: {path}")

print(f"Found Objective-1 and coordinate-head predictions for {len(sample_ids)} test objects")

In [ ]:
def raw_gt_path(sample_id):
    category, object_id = sample_id.split("__", 1)
    category = "cars" if category == "car" else category
    return RAW_SHAPENET_DIR / category / object_id / "models" / "model_normalized.obj"


def canonicalize_raw_gt(path):
    loaded = trimesh.load(path, process=False)
    mesh = loaded.to_mesh() if isinstance(loaded, trimesh.Scene) else loaded
    points = np.asarray(mesh.vertices, dtype=np.float64)
    points = np.column_stack((points[:, 0], -points[:, 2], points[:, 1]))
    bounds_min, bounds_max = points.min(axis=0), points.max(axis=0)
    points = (points - (bounds_min + bounds_max) / 2.0) / (bounds_max - bounds_min).max()
    faces = np.column_stack((np.full(len(mesh.faces), 3), mesh.faces)).ravel()
    return pv.PolyData(points, faces)


def load_ground_truth(sample_id):
    copied_path = RESULTS_DIR / "ground_truth" / "mesh" / f"{sample_id}.ply"
    canonical_path = DATASET_DIR / "renders" / sample_id / "mesh.ply"
    for path in (copied_path, canonical_path):
        if path.is_file():
            return pv.read(path)

    path = raw_gt_path(sample_id)
    if path.is_file():
        return canonicalize_raw_gt(path)
    raise FileNotFoundError(
        f"Missing ground truth for {sample_id}. Expected {canonical_path} or {path}"
    )


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_comparison(sample_id):
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    meshes = {"Ground truth": load_ground_truth(sample_id)}
    meshes.update({
        label: pv.read(mesh_dir / f"{sample_id}.ply")
        for label, mesh_dir in mesh_dirs.items()
    })

    # Define one cut and camera from GT, then reuse them for all three meshes.
    ground_truth = meshes["Ground truth"]
    center = np.asarray(ground_truth.center)
    projection = np.asarray(ground_truth.points) @ normal
    plane_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    cut_origin = normal * plane_offset
    object_size = ground_truth.length

    plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 650))
    for index, (label, _) in enumerate(METHODS):
        cut_mesh = meshes[label].clip(normal=normal, origin=cut_origin, invert=True)

        plotter.subplot(0, index)
        plotter.set_background("white")
        plotter.add_mesh(
            cut_mesh,
            color="lightsteelblue",
            smooth_shading=True,
            ambient=0.25,
            diffuse=0.8,
            specular=0.15,
        )
        plotter.add_text(label, position="upper_left", color="black", font_size=12)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.42 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)

In [ ]:
for sample_id in sample_ids:
    print(sample_id)
    display(render_comparison(sample_id))